In [1]:
import pandas as pd
import numpy as np

# Загружаем только то, что нужно
members = pd.read_csv('../data/raw/members_v3.csv')

# 1. Возраст 0 — это пропуск. Заменяем на NaN
members['bd'] = members['bd'].replace(0, np.nan)

# 2. Создаем бинарный признак пола: 1 - мужчина, 0 - женщина, NaN - не указан
members['is_male'] = members['gender'].map({'male': 1, 'female': 0})

# 3. Оставляем только нужные колонки для финального датасета
members_clean = members[['msno', 'city', 'bd', 'registered_via', 'is_male']]

In [2]:
transactions = pd.read_csv('../data/raw/transactions_v2.csv')

# Группируем по пользователю и считаем агрегаты
trans_features = transactions.groupby('msno').agg(
    total_paid=('actual_amount_paid', 'sum'),         # Всего заплатил
    avg_paid=('actual_amount_paid', 'mean'),          # Средний чек
    total_transactions=('transaction_date', 'count'), # Сколько раз платил
    total_cancel=('is_cancel', 'sum'),                # Сколько раз отменял
    auto_renew_mean=('is_auto_renew', 'mean'),        # Доля автопродлений (от 0 до 1)
    max_plan_days=('payment_plan_days', 'max')        # Максимальный купленный период (в днях)
).reset_index()

In [3]:
user_logs = pd.read_csv('../data/raw/user_logs_v2.csv')

# Группируем по пользователю
logs_features = user_logs.groupby('msno').agg(
    total_secs=('total_secs', 'sum'),       # Общее время прослушивания
    total_songs=('num_25', 'sum'),          # Сколько песен прослушал (упрощенно)
    avg_num_unq=('num_unq', 'mean')         # В среднем уникальных песен в день
).reset_index()

In [4]:
train = pd.read_csv('../data/raw/train_v2.csv')

# Объединяем (merge) таблицы по ключу 'msno'
# Используем left join, так как в train есть все нужные пользователи
df_final = train.merge(members_clean, on='msno', how='left') \
                 .merge(trans_features, on='msno', how='left') \
                 .merge(logs_features, on='msno', how='left')

print("Размер финального датасета:", df_final.shape)
print("Пропуски после объединения:\n", df_final.isnull().sum())

Размер финального датасета: (970960, 15)
Пропуски после объединения:
 msno                       0
is_churn                   0
city                  109993
bd                    583722
registered_via        109993
is_male               582055
total_paid             37382
avg_paid               37382
total_transactions     37382
total_cancel           37382
auto_renew_mean        37382
max_plan_days          37382
total_secs            216409
total_songs           216409
avg_num_unq           216409
dtype: int64


In [5]:
# 1. Для поведенческих и финансовых признаков заполняем пропуски нулями
# Если данных нет, значит активности не было.
features_to_zero = ['total_paid', 'avg_paid', 'total_transactions', 'total_cancel', 
                    'auto_renew_mean', 'max_plan_days', 'total_secs', 'total_songs', 'avg_num_unq']

df_final[features_to_zero] = df_final[features_to_zero].fillna(0)

# 2. Для города и пола создаем категорию "Неизвестно"
df_final['city'] = df_final['city'].fillna(-1).astype(int)
df_final['is_male'] = df_final['is_male'].fillna(-1).astype(int)

# 3. Для возраста оставляем NaN, но проверим, что структура правильная
# (bd уже имеет тип float из-за NaN, это нормально)

# Проверяем, что все пропуски обработаны (кроме bd)
print("Осталось пропусков:\n", df_final.isnull().sum())

Осталось пропусков:
 msno                       0
is_churn                   0
city                       0
bd                    583722
registered_via        109993
is_male                    0
total_paid                 0
avg_paid                   0
total_transactions         0
total_cancel               0
auto_renew_mean            0
max_plan_days              0
total_secs                 0
total_songs                0
avg_num_unq                0
dtype: int64


In [6]:
# Сохраняем в папку processed
# index=False нужен, чтобы не сохранять индексы строк pandas в CSV
df_final.to_csv('../data/processed/kkbox_features.csv', index=False)
print("Файл сохранен в data/processed/kkbox_features.csv")

Файл сохранен в data/processed/kkbox_features.csv
